# Segmentación de tumores cerebrales en MRI con TDA
## Descripción de los datos

## Descripción del Conjunto de Datos

El conjunto de datos utilizado consiste en imágenes de resonancia magnética (MRI) cerebral, preprocesadas y almacenadas en formato HDF5 (.h5). Cada muestra corresponde a un volumen 3D cerebral con **4 canales** de información, representando distintas modalidades de imagen médica (T1, T1ce, T2 y FLAIR), utilizadas comúnmente en tareas de segmentación de tumores cerebrales. El conjunto completo contiene **369 resonancias cerebrales**, cada una de las cuales ha sido seccionada en cortes axiales individuales. Estos cortes han sido organizados en un total de **155 archivos HDF5 por resonancia**.

Cada canal de imagen corresponde a una modalidad diferente de resonancia magnética:
1. **T1**: Imagen ponderada en T1 (usada para observar la anatomía general del cerebro).
2. **T1ce**: Imagen ponderada en T1 con contraste (utilizada para observar mejor los tumores y la lesión cerebral).
3. **T2**: Imagen ponderada en T2 (usada para identificar zonas de edema o inflamación).
4. **FLAIR**: Imagen ponderada en FLAIR (utilizada para resaltar lesiones cerebrales y tumores, especialmente en el área de la sustancia blanca).

Este dataset se tomó de [Kaggle](https://www.kaggle.com/datasets/awsaf49/brats2020-training-data?select=BraTS20+Training+Metadata.csv) y proviene del desafío **BraTS2020 (Brain Tumor Segmentation año 2020)**, ampliamente utilizado para la evaluación de modelos de aprendizaje profundo en tareas médicas de segmentación 3D.

In [4]:
import h5py
import numpy as np
import os

# Ruta a los archivos .h5 (suponiendo que los cortes están en una carpeta)
folder_path = 'data/BraTS2020_training_data/content/data'  # Ruta a la carpeta donde están los archivos .h5
files = os.listdir(folder_path) # Obtener la lista de archivos

# Verifica que todos los archivos sean .h5 y tomamos solo los primeros 158 archivos9
h5_files = [f for f in files[0:157] if f.endswith('.h5')]

# Ordenar los archivos primero por volumen y luego por slice
sorted_h5_files = sorted(h5_files, key=lambda f: (int(f.split('_')[1]), int(f.split('_')[3].split('.')[0])))

# Listado de cortes 2D
volume_data_brain = []
volume_data_tumor = []

# Cargar cada archivo .h5 y extraer el corte 2D
for f in sorted_h5_files:
    with h5py.File(os.path.join(folder_path, f), 'r') as file:
        # La imagen 2D está almacenada en la clave 'image', el tumor en la clave 'mask'.
        brain_slice = file['image'][:]
        tumor_slice = file['mask'][:]

        volume_data_brain.append(brain_slice)
        volume_data_tumor.append(tumor_slice)

# Apilar los cortes 2D a lo largo del eje Z (creando el volumen 3D)
volume_3d_brain = np.stack(volume_data_brain, axis=-1)
volume_3d_tumor = np.stack(volume_data_tumor, axis=-1)

print("Forma del volumen 3D del cerebro:", volume_3d_brain.shape)
print("Forma del volumen 3D del tumor:", volume_3d_tumor.shape)

Forma del volumen 3D del cerebro: (240, 240, 4, 154)
Forma del volumen 3D del tumor: (240, 240, 3, 154)


In [12]:
import pandas as pd

volume_brain_for_pandas = np.array([np.ravel(volume_3d_brain[:,:,channel,:]) for channel in range(volume_3d_brain.shape[2])])
pd.DataFrame(volume_brain_for_pandas.T, columns=['Channel 0', 'Channel 1', 'Channel 2', 'Channel 3'])

,Channel 0,Channel 1,Channel 2,Channel 3
0,0.000000,0.000000,0.000000,0.000000
1,-0.008317,-0.008332,-0.008313,-0.008319
2,-0.023950,-0.024202,-0.024175,-0.024235
3,-0.032637,-0.032627,-0.032519,-0.032529
4,-0.044941,-0.041330,-0.042337,-0.046971
...,...,...,...,...
8870395,0.000000,0.000000,0.000000,0.000000
8870396,0.000000,0.000000,0.000000,0.000000
8870397,0.000000,0.000000,0.000000,0.000000
8870398,0.000000,0.000000,0.000000,0.000000


In [14]:
volume_tumor_for_pandas = np.array([np.ravel(volume_3d_tumor[:,:,channel,:]) for channel in range(volume_3d_tumor.shape[2])])
pd.DataFrame(volume_tumor_for_pandas.T, columns=['Channel 0', 'Channel 1', 'Channel 3']).describe()

,Channel 0,Channel 1,Channel 3
count,8.870400e+06,8.870400e+06,8.870400e+06
mean,5.319264e-03,6.417185e-03,3.383726e-03
std,7.273905e-02,7.984989e-02,5.807130e-02
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00,0.000000e+00
max,1.000000e+00,1.000000e+00,1.000000e+00


In [67]:
import napari

# volumen_3d: (240, 240, 4, 154) del cerebro
# napari espera el volumen como (profundidad, altura, ancho)
volume_brain_for_napari = np.moveaxis(volume_3d_brain, [2, 3], [0, 1])  # (4, 154, 240, 240)

# Crear el visor de Napari en 3D (ndisplay=3)
viewer_3d = napari.Viewer(ndisplay=3)

# Añadir la imagen 3D con renderizado isovolumétrico para el modelo 3D
for i, vol in enumerate(volumen_para_napari):
    layer = viewer_3d.add_image(
        data=vol,
        name=f'Canal {i}',
        colormap='gray',
        rendering='iso',
        iso_threshold=1.5,
        contrast_limits=[1, 10],
        blending='opaque'
    )

napari.run()

In [68]:
# Crear el visor de Napari por slices
viewer_2d = napari.Viewer(ndisplay=2)

# Añadir la imagen 2D para las slices (en cada canal) con renderizado normal
for i, vol in enumerate(volumen_para_napari):
    layer = viewer_2d.add_image(
        data=vol,
        name=f'Canal {i}',
        colormap='inferno',
        rendering='translucent',
    )
    layer.reset_contrast_limits()  # Activar autocontraste continuo

napari.run()
